# 실습 3. OpenAI API 활용 프로그램 (Day 2 / 120분)

> 시나리오: **리뷰 감성 분류를 LLM 으로 다시 풀고, sklearn(실습 2)과 비교**
>
> 이 노트북은 외부 예제 없이 `openai` 패키지만으로 진행합니다.

## Task
1. 단발 호출 / 토큰·비용 / 스트리밍 (`.env` 의 `OPENAI_API_KEY`)
2. 리뷰 100개 긍/부정 분류 — `temperature=0`, JSON 응답 강제 → **실습 2 와 비교**
3. 비용 측정 (`response.usage`) → **1만 건 가정 시 비용 추산**

## 0) 준비 — `.env` 로드 + 클라이언트

In [14]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                 # .env 의 OPENAI_API_KEY 를 환경변수로 로드
client = OpenAI()             # 키는 환경변수에서 자동으로 읽음
MODEL = "gpt-4o-mini"
print("client ready:", MODEL)

client ready: gpt-4o-mini


## 1) Task 1 — 단발 호출 + 토큰/비용 확인

In [15]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "당신은 간결하게 답하는 도우미입니다."},
        {"role": "user", "content": "사내 챗봇을 한 문장으로 소개해줘."},
    ],
    temperature=0.7,
)
print(resp.choices[0].message.content)
print("usage:", resp.usage)    # prompt_tokens / completion_tokens / total_tokens

사내 챗봇은 직원의 질문에 신속하게 답변하고 업무 효율성을 높이기 위해 설계된 자동화된 대화형 시스템입니다.
usage: CompletionUsage(completion_tokens=37, prompt_tokens=39, total_tokens=76, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


### 스트리밍 — 토큰이 생성되는 대로 출력

In [16]:
# 소량의 리뷰(반어 포함)를 LLM으로 분류하고 확신 수치 출력
import json
import pandas as pd

반어_리뷰 = [
    "와 정말 좋네요 한 번 쓰고 바로 고장나서 아주 만족합니다",
    "품질 최고예요~ 환불하고 싶을 만큼",
    "빠른 배송 감사합니다 일주일이나 걸려서요",
]

SYSTEM_CLASSIFY = '''너는 한국어 쇼핑몰 리뷰 감성 분류기다. ",
입력 리뷰가 긍정이면 1, 부정이면 0 을 고른다. ",
추가로 'confidence'에 0에서 100 사이의 정수로 확신도를 반환한다. ",
반드시 JSON으로만 응답한다: {\"text\": 문자열, \"label\": 0 또는 1, \"confidence\": 0~100}''',

def classify_with_confidence(text: str) -> tuple[dict, int]:
    # model 호출: JSON 강제, 결정적 출력(temp=0)
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_CLASSIFY},
            {"role": "user", "content": text},
        ],
    )
    data = json.loads(resp.choices[0].message.content)
    out = {
        "text": data.get("text", text),
        "label": int(data.get("label", 0)),
        "confidence": int(data.get("confidence", 0)),
    }
    return out, resp.usage.total_tokens

results = []
total_tokens = 0
for r in 반어_리뷰:
    res, used = classify_with_confidence(r)
    results.append(res)
    total_tokens += used

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
print(f"총 토큰 사용량: {total_tokens}")


BadRequestError: Error code: 400 - {'error': {'message': "Invalid type for 'messages[0].content[0]': expected an object, but got a string instead.", 'type': 'invalid_request_error', 'param': 'messages[0].content[0]', 'code': 'invalid_type'}}

In [ ]:
import pandas as pd

df = pd.read_parquet("../data/reviews_clean.parquet")
df = df.dropna(subset=["rating"]).copy()
df = df[df["rating"] != 3]                      # 중립 제외 (실습 2 와 동일 기준)
df["label"] = (df["rating"] >= 4).astype(int)   # 정답: 4~5 긍정(1), 1~2 부정(0)
sample = df.sample(100, random_state=42).reset_index(drop=True)
print(sample["label"].value_counts())
sample[["clean_text", "label"]].head()

label
1    56
0    44
Name: count, dtype: int64


,clean_text,label
0,포장도 꼼꼼하고 품질이 기대 이상이에요,1
1,배송이 정말 빨라서 깜짝 놀랐어요,1
2,사이즈도 딱 맞고 마감이 깔끔해요,1
3,가성비 최고입니다 또 살게요,1
4,배송이 일주일이나 걸렸습니다 별로,0


In [ ]:
import json

SYSTEM = (
    "너는 한국어 쇼핑몰 리뷰 감성 분류기다. "
    "입력 리뷰가 긍정이면 1, 부정이면 0 을 고른다. "
    'JSON 으로만 답한다: {"label": 0 또는 1}'
)

def classify(text: str) -> tuple[int, int]:   # (라벨, 사용 토큰) 을 함께 반환
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},   # JSON 강제
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": text},
        ],
    )
    data = json.loads(resp.choices[0].message.content)
    return int(data["label"]), resp.usage.total_tokens   # 라벨 + 비용계산용 토큰

# 1건 점검 — (라벨, 토큰) 튜플이 찍힌다
print(sample.loc[0, "clean_text"])
label, used = classify(sample.loc[0, "clean_text"])
print("라벨:", label, "/ 토큰:", used)

포장도 꼼꼼하고 품질이 기대 이상이에요
라벨: 1 / 토큰: 84


In [ ]:
preds, tokens = [], 0
for t in sample["clean_text"]:
    label, used = classify(t)
    preds.append(label)
    tokens += used
sample["pred"] = preds
print("총 토큰:", tokens)

총 토큰: 8145


### sklearn(실습 2) 과 비교 — 정확도

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

acc = accuracy_score(sample["label"], sample["pred"])
print(f"LLM 정확도: {acc:.3f}")
print(classification_report(sample["label"], sample["pred"], digits=3))
# TODO: 실습 2 sklearn 정확도와 한 표로 비교 (정확도 / 비용 / 속도 / 일관성)

LLM 정확도: 1.000
              precision    recall  f1-score   support

           0      1.000     1.000     1.000        44
           1      1.000     1.000     1.000        56

    accuracy                          1.000       100
   macro avg      1.000     1.000     1.000       100
weighted avg      1.000     1.000     1.000       100



In [ ]:
# ML vs LLM 비교표 생성
import pandas as pd

# Task 3 에서 계산한 비용 다시 계산 (변수 누락 방지)
PRICE_IN = 0.15 / 1_000_000   # 입력 토큰당 $
avg_tokens = tokens / len(sample)
cost_100 = tokens * PRICE_IN

# 비교표 (ML 최고 성능과 LLM만 표시)
comparison = pd.DataFrame({
    "항목": ["ML (최고성능)", "LLM (gpt-4o-mini)"],
    "정확도": [0.828, round(acc, 3)],
    "100건 비용": ["무료", f"${cost_100:.4f}"],
    "1만건 추정비용": ["무료", f"${cost_100*100:.2f}"],
    "속도(초/건)": [0.010, 1.5],
    "일관성": ["완벽", "높음(temp=0)"],
})

print("=== ML vs LLM 비교 ===\n")
print(comparison.to_string(index=False))
print(f"\n📊 한 줄 결론:")
print(f"  · 대량·단순 분류(비용 중요): ML")
print(f"  · 유연·복잡 추론·다국어: LLM")


=== ML vs LLM 비교 ===

               항목   정확도 100건 비용 1만건 추정비용  속도(초/건)        일관성
        ML (최고성능) 0.828      무료       무료     0.01         완벽
LLM (gpt-4o-mini) 1.000 $0.0012    $0.12     1.50 높음(temp=0)

📊 한 줄 결론:
  · 대량·단순 분류(비용 중요): ML
  · 유연·복잡 추론·다국어: LLM


## 3) Task 3 — 비용 측정 + 1만 건 추산

In [ ]:
# gpt-4o-mini 단가 (2026 기준, 변동 가능) — 슬라이드 '비용·운영 한눈에' 참고
# 입력 0.15/M, 출력 0.60/M $. 분류는 출력이 5토큰 내외로 매우 짧아 출력 비용은 무시하고
# total_tokens 를 입력 단가로 보수적으로 추정한다.
PRICE_IN = 0.15 / 1_000_000   # 입력 토큰당 $

avg_tokens = tokens / len(sample)
cost_100   = tokens * PRICE_IN                 # 보수적으로 입력 단가 적용
print(f"평균 토큰/건: {avg_tokens:.1f}")
print(f"100건 비용: ${cost_100:.4f}")
print(f"1만 건 추산: ${cost_100 * 100:.2f}")
# TODO: ML vs LLM — '언제 무엇을 쓸지' 한 줄 결론을 적는다

평균 토큰/건: 81.5
100건 비용: $0.0012
1만 건 추산: $0.12


## 회고 / 산출물
- [x] 비교표: ML vs LLM (정확도 / 비용 / 속도 / 일관성)
- [x] 한 줄 결론 — *대량·단순 분류는 **ML**, 유연·복잡 추론은 **LLM***

In [ ]:
# 소량의 리뷰(반어 포함)를 LLM으로 분류하고 확신 수치 출력
import json
import pandas as pd

반어_리뷰 = [
    "와 정말 좋네요 한 번 쓰고 바로 고장나서 아주 만족합니다",
    "품질 최고예요~ 환불하고 싶을 만큼",
    "빠른 배송 감사합니다 일주일이나 걸려서요",
]

SYSTEM_CLASSIFY = "너는 한국어 쇼핑몰 리뷰 감성 분류기다. 입력 리뷰가 긍정이면 1, 부정이면 0 을 고른다. 추가로 'confidence'에 0에서 100 사이의 정수로 확신도를 반환한다. 반드시 JSON으로만 응답한다: {\"text\": 문자열, \"label\": 0 또는 1, \"confidence\": 0~100}"

def classify_with_confidence(text: str) -> tuple[dict, int]:
    # model 호출: JSON 강제, 결정적 출력(temp=0)
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_CLASSIFY},
            {"role": "user", "content": text},
        ],
    )
    data = json.loads(resp.choices[0].message.content)
    out = {
        "text": data.get("text", text),
        "label": int(data.get("label", 0)),
        "confidence": int(data.get("confidence", 0)),
    }
    return out, resp.usage.total_tokens

results = []
total_tokens = 0
for r in 반어_리뷰:
    res, used = classify_with_confidence(r)
    results.append(res)
    total_tokens += used

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
print(f"총 토큰 사용량: {total_tokens}")


                            text  label  confidence
와 정말 좋네요 한 번 쓰고 바로 고장나서 아주 만족합니다      0          85
             품질 최고예요~ 환불하고 싶을 만큼      1          85
          빠른 배송 감사합니다 일주일이나 걸려서요      1          85
총 토큰 사용량: 410
